# 02. Machine Learning Model Training & Comparison

**Project**: Customer Churn Prediction System
**Objective**: Train and benchmark multiple classification algorithms on the cleaned Telco dataset, evaluate through 5-fold cross-validation, and serialize the best performing model pipeline.

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, AdaBoostClassifier

import sys
sys.path.append("..")
from src.features import build_features
from src.config import NUMERIC_COLUMNS, RANDOM_STATE, TEST_SIZE, CV_FOLDS, CANONICAL_FEATURE_COLUMNS

## 1. Load Cleaned Dataset & Prepare Target/Features

In [2]:
df = pd.read_csv("../Dataset/cleaned_telco_churn.csv")
print("Dataset Shape:", df.shape)

y = (df["Churn"] == "Yes").astype(int)
X = build_features(df)

print("Feature Matrix X:", X.shape)
print("Target y distribution:", y.value_counts().to_dict())

Dataset Shape: (7043, 21)
Feature Matrix X: (7043, 30)
Target y distribution: {0: 5174, 1: 1869}


## 2. Stratified Train / Test Split (80 / 20)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

Training set: (5634, 30), Test set: (1409, 30)


## 3. Build Preprocessing & Classification Pipelines

In [4]:
preprocessor = ColumnTransformer(
    transformers=[("num", StandardScaler(), NUMERIC_COLUMNS)],
    remainder="passthrough"
)

models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(C=0.5, class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "Decision Tree": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(n_estimators=250, max_depth=8, min_samples_split=10, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1))
    ]),
    "Gradient Boosting": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, max_depth=3, subsample=0.85, random_state=RANDOM_STATE))
    ]),
    "HistGradientBoosting": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", HistGradientBoostingClassifier(max_iter=150, learning_rate=0.08, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "AdaBoost": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=RANDOM_STATE))
    ])
}
print(f"Configured {len(models)} candidate pipelines.")

Configured 6 candidate pipelines.


## 4. Run 5-Fold Stratified Cross-Validation

In [5]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
scoring = ["roc_auc", "average_precision", "recall", "f1", "accuracy"]
cv_results = {}

for name, pipe in models.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_results[name] = {
        "CV ROC-AUC": f"{np.mean(res['test_roc_auc']):.4f} +/- {np.std(res['test_roc_auc']):.4f}",
        "CV Recall": f"{np.mean(res['test_recall']):.4f}",
        "CV F1": f"{np.mean(res['test_f1']):.4f}"
    }

pd.DataFrame(cv_results).T

,CV ROC-AUC,CV Recall,CV F1
Logistic Regression,0.8459 +/- 0.0124,0.8013,0.6296
Decision Tree,0.8250 +/- 0.0091,0.7846,0.6030
Random Forest,0.8469 +/- 0.0116,0.7893,0.6326
Gradient Boosting,0.8488 +/- 0.0111,0.5304,0.5916
HistGradientBoosting,0.8359 +/- 0.0109,0.7284,0.6247
AdaBoost,0.8451 +/- 0.0122,0.4495,0.5414


## 5. Fit & Evaluate All Models on Held-out Test Set

In [6]:
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, precision_score, f1_score, accuracy_score

test_records = []
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.45).astype(int)
    
    test_records.append({
        "Model": name,
        "Test ROC-AUC": round(roc_auc_score(y_test, y_prob), 4),
        "Test PR-AUC": round(average_precision_score(y_test, y_prob), 4),
        "Test Recall": round(recall_score(y_test, y_pred), 4),
        "Test F1": round(f1_score(y_test, y_pred), 4),
        "Test Accuracy": round(accuracy_score(y_test, y_pred), 4)
    })

comparison_df = pd.DataFrame(test_records).sort_values(by="Test ROC-AUC", ascending=False)
comparison_df

,Model,Test ROC-AUC,Test PR-AUC,Test Recall,Test F1,Test Accuracy
3,Gradient Boosting,0.8441,0.6599,0.5989,0.6162,0.8020
2,Random Forest,0.8435,0.6541,0.8342,0.6246,0.7339
0,Logistic Regression,0.8419,0.6344,0.8369,0.6131,0.7197
5,AdaBoost,0.8408,0.6507,0.6765,0.6111,0.7715
4,HistGradientBoosting,0.8365,0.6436,0.7834,0.6221,0.7473
1,Decision Tree,0.8298,0.6009,0.8529,0.6111,0.7119
